In [2]:
import pandas as pd
import requests
from typing import List

def fetch_uniprot_info(uniprot_ids: List[str]) -> pd.DataFrame:
    """
    Queries UniProt for a list of UniProt accessions and retrieves the protein name and PDB cross-references.

    Args:
        uniprot_ids (List[str]): A list of UniProt IDs to query.

    Returns:
        pd.DataFrame: A DataFrame with columns ['Entry', 'Protein_Name', 'PDB_IDs'].
    """
    results = []

    for uid in uniprot_ids:
        try:
            # Query UniProt REST API for a single UniProt ID
            url = f"https://rest.uniprot.org/uniprotkb/{uid}.json"
            response = requests.get(url, timeout=10)

            # Check for success
            if response.status_code == 200:
                data = response.json()

                # Extract protein name (from recommendedName)
                protein_name = data.get('proteinDescription', {}).get('recommendedName', {}).get('fullName', {}).get('value', 'N/A')

                # Extract PDB cross-references
                pdb_ids = [
                    x['id'] for x in data.get('uniProtKBCrossReferences', [])
                    if x['database'] == 'PDB'
                ]
                pdb_ids_str = ';'.join(pdb_ids) if pdb_ids else 'None'

                results.append({
                    'Entry': uid,
                    'Protein_Name': protein_name,
                    'PDB_IDs': pdb_ids_str
                })
            else:
                results.append({
                    'Entry': uid,
                    'Protein_Name': 'Not found',
                    'PDB_IDs': 'Not found'
                })
        except Exception as e:
            results.append({
                'Entry': uid,
                'Protein_Name': f'Error: {str(e)}',
                'PDB_IDs': 'Error'
            })

    return pd.DataFrame(results)


def annotate_uniprot_with_pdb(input_csv: str, output_csv: str) -> None:
    """
    Annotates a CSV file containing a column 'Entry' with protein names and PDB IDs from UniProt.

    Args:
        input_csv (str): Path to the input CSV file with UniProt IDs in 'Entry' column.
        output_csv (str): Path to write the annotated CSV output.
    """
    # Read the input CSV
    df = pd.read_csv(input_csv)

    # Ensure 'Entry' column exists
    if 'Entry' not in df.columns:
        raise ValueError("The input CSV must contain a column named 'Entry'.")

    # Get unique UniProt IDs
    unique_ids = df['Entry'].dropna().unique().tolist()

    # Query UniProt
    annotations_df = fetch_uniprot_info(unique_ids)

    # Merge annotations into original DataFrame
    merged_df = pd.merge(df, annotations_df, on='Entry', how='left')

    # Save the annotated file
    merged_df.to_csv(output_csv, index=False)
    print(f"Annotated file saved to: {output_csv}")


# === USAGE ===
annotate_uniprot_with_pdb(
    input_csv="submission/Book_AJ_attempt_4.csv",
    output_csv="submission/Book_AJ_attempt_4_annotated.csv"
)


Annotated file saved to: submission/Book_AJ_attempt_4_annotated.csv
